# Rebuild diff_text from local clones

Runs `scripts/build_diffs.py` and streams its log/progress into this notebook.

- No GitHub / network — reads only the cloned repos under `./repos/apache/`
- Output: `data/apachejit/apachejit_with_diffs_rebuilt.csv` (all feature columns + fixed `diff_text` **with** `--- a/ +++ b/` headers)
- Resumable: re-run this cell and it skips commits already done

Optional: add `"--project", "apache/groovy"` to the arg list to test one project first.

In [ ]:
import sys, subprocess

proc = subprocess.Popen(
    [sys.executable, "-u", "scripts/build_diffs.py",
     "--csv",   "data/apachejit/apachejit_total.csv",
     "--repos", "./repos/apache/",
     "--out",   "data/apachejit/apachejit_with_diffs_rebuilt.csv"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, encoding="utf-8", errors="replace", bufsize=1)

for line in proc.stdout:
    print(line, end="")
proc.wait()

## Verify the rebuilt file

Confirms the `--- a/ +++ b/` headers are now present (the check that failed on the old reconciled file).

In [ ]:
import pandas as pd

d = pd.read_csv("data/apachejit/apachejit_with_diffs_rebuilt.csv",
                usecols=["commit_id", "project", "diff_text"], nrows=200)

def has(s, sub): return isinstance(s, str) and sub in s
n = len(d)
print(f"rows checked: {n}")
for marker in ["commit ", "Author: ", "--- a/", "+++ b/", "@@ "]:
    c = d.diff_text.apply(lambda s: has(s, marker)).sum()
    print(f"  {marker!r:12} present in {c}/{n} rows")

s = d.diff_text.dropna().iloc[0]
print("\n--- first 600 chars of diff_text[0] ---")
print(repr(s[:600]))